In [2]:
!uname -a && cat /etc/*release

Linux 25ef0b8ac592 6.6.113+ #1 SMP Mon Feb  2 12:27:57 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
DISTRIB_ID=Ubuntu
DISTRIB_RELEASE=22.04
DISTRIB_CODENAME=jammy
DISTRIB_DESCRIPTION="Ubuntu 22.04.5 LTS"
PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy


In [3]:
!pwd
!ls -la

/content
total 24
drwxr-xr-x 1 root root 4096 May  7 18:22 .
drwxr-xr-x 1 root root 4096 May  7 18:05 ..
drwxr-xr-x 4 root root 4096 Apr 16 13:33 .config
-rw-r--r-- 1 root root 4500 May  7 18:22 MatMulCuda.cu
drwxr-xr-x 1 root root 4096 Apr 16 13:33 sample_data


In [4]:
!nvidia-smi

Thu May  7 18:22:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!nvcc -arch=sm_75 -O3 MatMulCuda.cu -o MatMulCuda_fp64
!./MatMulCuda_fp64

MatMulCuda (naive) -- n=10000, dtype=double (FP64), block=16x16
Matrix memory per buffer: 762.94 MB
C[0][0]   = 60000.000000 (expected 60000.000000, abs err 0.000e+00)
C[n-1][n-1] = 60000.000000

--- Timings (ms) ---
H2D copy   :    346.305 ms
Kernel     :   9048.547 ms
D2H copy   :    600.066 ms
Total GPU  :   9994.918 ms  (9.995 s)


In [6]:
!nvcc -arch=sm_75 -O3 -DUSE_FLOAT MatMulCuda.cu -o MatMulCuda_fp32
!./MatMulCuda_fp32

MatMulCuda (naive) -- n=10000, dtype=float (FP32), block=16x16
Matrix memory per buffer: 381.47 MB
C[0][0]   = 60000.000000 (expected 60000.000000, abs err 0.000e+00)
C[n-1][n-1] = 60000.000000

--- Timings (ms) ---
H2D copy   :    172.943 ms
Kernel     :   6200.314 ms
D2H copy   :    301.300 ms
Total GPU  :   6674.558 ms  (6.675 s)


In [7]:
# coalescing / efficiency metrics (FP64). Slow because of profiler overhead, that's OK.
!nvprof --metrics gld_efficiency,gst_efficiency,gld_throughput,gst_throughput,sm_efficiency ./MatMulCuda_fp64

======== Warning: Skipping profiling on device 0 since profiling is not supported on devices with compute capability 7.5 and higher.
                  Use NVIDIA Nsight Compute for GPU profiling and NVIDIA Nsight Systems for GPU tracing and CPU sampling.
                  Refer https://developer.nvidia.com/tools-overview for more details.

MatMulCuda (naive) -- n=10000, dtype=double (FP64), block=16x16
Matrix memory per buffer: 762.94 MB
==9909== NVPROF is profiling process 9909, command: ./MatMulCuda_fp64
C[0][0]   = 60000.000000 (expected 60000.000000, abs err 0.000e+00)
C[n-1][n-1] = 60000.000000

--- Timings (ms) ---
H2D copy   :    342.739 ms
Kernel     :   9146.360 ms
D2H copy   :    539.282 ms
Total GPU  :  10028.381 ms  (10.028 s)
==9909== Profiling application: ./MatMulCuda_fp64
==9909== Profiling result:
No events/metrics were profiled.


In [8]:
# Cell F - same metrics for FP32
!nvprof --metrics gld_efficiency,gst_efficiency,gld_throughput,gst_throughput,sm_efficiency ./MatMulCuda_fp32

======== Warning: Skipping profiling on device 0 since profiling is not supported on devices with compute capability 7.5 and higher.
                  Use NVIDIA Nsight Compute for GPU profiling and NVIDIA Nsight Systems for GPU tracing and CPU sampling.
                  Refer https://developer.nvidia.com/tools-overview for more details.

MatMulCuda (naive) -- n=10000, dtype=float (FP32), block=16x16
Matrix memory per buffer: 381.47 MB
==10126== NVPROF is profiling process 10126, command: ./MatMulCuda_fp32
C[0][0]   = 60000.000000 (expected 60000.000000, abs err 0.000e+00)
C[n-1][n-1] = 60000.000000

--- Timings (ms) ---
H2D copy   :    174.360 ms
Kernel     :   6109.854 ms
D2H copy   :    317.604 ms
Total GPU  :   6601.818 ms  (6.602 s)
==10126== Profiling application: ./MatMulCuda_fp32
==10126== Profiling result:
No events/metrics were profiled.


## 2: block-size sweep on the naive kernel

### FP64

In [10]:
# FP64 sweep
!nvcc -arch=sm_75 -O3 -DBLOCK=8  MatMulCuda.cu -o MatMulCuda_fp64_b8
!nvcc -arch=sm_75 -O3 -DBLOCK=16 MatMulCuda.cu -o MatMulCuda_fp64_b16
!nvcc -arch=sm_75 -O3 -DBLOCK=32 MatMulCuda.cu -o MatMulCuda_fp64_b32

!nvprof ./MatMulCuda_fp64_b8
!nvprof ./MatMulCuda_fp64_b16
!nvprof ./MatMulCuda_fp64_b32

MatMulCuda (naive) -- n=10000, dtype=double (FP64), block=8x8
Matrix memory per buffer: 762.94 MB
==9757== NVPROF is profiling process 9757, command: ./MatMulCuda_fp64_b8
C[0][0]   = 60000.000000 (expected 60000.000000, abs err 0.000e+00)
C[n-1][n-1] = 60000.000000

--- Timings (ms) ---
H2D copy   :    372.972 ms
Kernel     :  14099.133 ms
D2H copy   :    546.969 ms
Total GPU  :  15019.073 ms  (15.019 s)
==9757== Profiling application: ./MatMulCuda_fp64_b8
==9757== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   93.89%  14.0989s         1  14.0989s  14.0989s  14.0989s  matmul_naive(double const *, double const *, double*, int)
                    3.63%  545.64ms         1  545.64ms  545.64ms  545.64ms  [CUDA memcpy DtoH]
                    2.48%  372.41ms         2  186.21ms  180.45ms  191.96ms  [CUDA memcpy HtoD]
      API calls:   98.62%  15.0187s         3  5.00623s  180.69ms  14.6458s  cudaMemcpy
              

### FP32

In [11]:
# FP32 sweep
!nvcc -arch=sm_75 -O3 -DUSE_FLOAT -DBLOCK=8  MatMulCuda.cu -o MatMulCuda_fp32_b8
!nvcc -arch=sm_75 -O3 -DUSE_FLOAT -DBLOCK=16 MatMulCuda.cu -o MatMulCuda_fp32_b16
!nvcc -arch=sm_75 -O3 -DUSE_FLOAT -DBLOCK=32 MatMulCuda.cu -o MatMulCuda_fp32_b32

! nvprof ./MatMulCuda_fp32_b8
!nvprof ./MatMulCuda_fp32_b16
!nvprof ./MatMulCuda_fp32_b32

MatMulCuda (naive) -- n=10000, dtype=float (FP32), block=8x8
Matrix memory per buffer: 381.47 MB
==10127== NVPROF is profiling process 10127, command: ./MatMulCuda_fp32_b8
C[0][0]   = 60000.000000 (expected 60000.000000, abs err 0.000e+00)
C[n-1][n-1] = 60000.000000

--- Timings (ms) ---
H2D copy   :    174.494 ms
Kernel     :   9717.572 ms
D2H copy   :    275.590 ms
Total GPU  :  10167.656 ms  (10.168 s)
==10127== Profiling application: ./MatMulCuda_fp32_b8
==10127== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   95.59%  9.71734s         1  9.71734s  9.71734s  9.71734s  matmul_naive(float const *, float const *, float*, int)
                    2.70%  274.51ms         1  274.51ms  274.51ms  274.51ms  [CUDA memcpy DtoH]
                    1.71%  173.98ms         2  86.992ms  85.879ms  88.105ms  [CUDA memcpy HtoD]
      API calls:   98.16%  10.1673s         3  3.38911s  86.126ms  9.99289s  cudaMemcpy
              

In [9]:
!nvprof --metrics gld_efficiency,gst_efficiency,gld_throughput,gst_throughput,sm_efficiency ./MatMulCuda_fp32_b8

======== Warning: Skipping profiling on device 0 since profiling is not supported on devices with compute capability 7.5 and higher.
                  Use NVIDIA Nsight Compute for GPU profiling and NVIDIA Nsight Systems for GPU tracing and CPU sampling.
                  Refer https://developer.nvidia.com/tools-overview for more details.

MatMulCuda (naive) -- n=10000, dtype=float (FP32), block=8x8
Matrix memory per buffer: 381.47 MB
==9252== NVPROF is profiling process 9252, command: ./MatMulCuda_fp32_b8
C[0][0]   = 60000.000000 (expected 60000.000000, abs err 0.000e+00)
C[n-1][n-1] = 60000.000000

--- Timings (ms) ---
H2D copy   :    171.949 ms
Kernel     :  10009.197 ms
D2H copy   :    315.741 ms
Total GPU  :  10496.887 ms  (10.497 s)
==9252== Profiling application: ./MatMulCuda_fp32_b8
==9252== Profiling result:
No events/metrics were profiled.


### nsight-compute (ncu)

In [ ]:
# Nsight Compute - replaces nvprof on T4. Pick a few key sections.
!ncu --launch-count 1 --section LaunchStats --section Occupancy --section MemoryWorkloadAnalysis ./MatMulCuda_fp32_b16
!ncu --launch-count 1 --section LaunchStats --section Occupancy --section MemoryWorkloadAnalysis ./MatMulCuda_fp32_b32

## Step 3 Tiling
- instead of relying on CPU L1 to hold the working set, we explicitly stage the tiles of A and B into the SM's shared memory.
- Every thread in the block reuses each loaded element tile times before the next tile is loaded.
- This drops global memory-traffic by a factor of tile versus the naive kernel.

In [12]:
# Build
!nvcc -arch=sm_75 -O3            MatMulCudaTiled.cu -o MatMulCudaTiled_fp64_t16
!nvcc -arch=sm_75 -O3 -DUSE_FLOAT MatMulCudaTiled.cu -o MatMulCudaTiled_fp32_t16

# Run with nvprof activity breakdown
!nvprof ./MatMulCudaTiled_fp64_t16
!nvprof ./MatMulCudaTiled_fp32_t16

MatMulCudaTiled -- n=10000, dtype=double (FP64), tile=16x16
Matrix memory per buffer: 762.94 MB
Shared memory per block:  4096 bytes (2 x 16 x 16 x 8)
==13866== NVPROF is profiling process 13866, command: ./MatMulCudaTiled_fp64_t16
C[0][0]   = 60000.000000 (expected 60000.000000, abs err 0.000e+00)
C[n-1][n-1] = 60000.000000

--- Timings (ms) ---
H2D copy   :    345.455 ms
Kernel     :   8309.403 ms
D2H copy   :    641.945 ms
Total GPU  :   9296.803 ms  (9.297 s)
==13866== Profiling application: ./MatMulCudaTiled_fp64_t16
==13866== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   89.40%  8.30920s         1  8.30920s  8.30920s  8.30920s  matmul_tiled(double const *, double const *, double*, int)
                    6.89%  640.41ms         1  640.41ms  640.41ms  640.41ms  [CUDA memcpy DtoH]
                    3.71%  344.97ms         2  172.49ms  171.16ms  173.81ms  [CUDA memcpy HtoD]
      API calls:   98.04%  9.29648

### NCU analysis:

In [13]:
# Optional - shared-memory traffic and L1/L2 hit rates with Nsight Compute
!ncu --launch-count 1 \
     --section MemoryWorkloadAnalysis \
     --section LaunchStats \
     --section Occupancy \
     ./MatMulCudaTiled_fp32_t16

!ncu --launch-count 1 \
     --section MemoryWorkloadAnalysis \
     --section LaunchStats \
     --section Occupancy \
     ./MatMulCudaTiled_fp64_t16

MatMulCudaTiled -- n=10000, dtype=float (FP32), tile=16x16
Matrix memory per buffer: 381.47 MB
Shared memory per block:  2048 bytes (2 x 16 x 16 x 4)
==PROF== Connected to process 14156 (/content/MatMulCudaTiled_fp32_t16)
==PROF== Profiling "matmul_tiled" - 0 (1/1): 0%..
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
..50%....100% - 8 passes
C[0][0]   = 60000.000000 (expected 60000.000000, abs err 0.000e+00)
C[n-1][n-1] = 60000.000000

--- Timings (ms) ---
H2D copy   :    185.690 ms
Kernel     :  43367.496 ms
D2H copy   :    304.357 ms
Total GPU  :  43857.543 ms  (43.858 s)
==PROF== Disconnected from process 14156
[14156] MatMulCudaTiled_fp32_t16@127.0.0.1
  matmul_tiled(const float *, const float *, float *, int) (625, 625, 1)x(16, 16, 

## 4: Tile-size sweep (8, 16, 32) on the tiled kernel

In [14]:
# n=10000 builds (TILE=8 and TILE=16)
!nvcc -arch=sm_75 -O3            -DTILE=8  MatMulCudaTiled.cu -o MatMulCudaTiled_fp64_t8
!nvcc -arch=sm_75 -O3            -DTILE=16 MatMulCudaTiled.cu -o MatMulCudaTiled_fp64_t16
!nvcc -arch=sm_75 -O3 -DUSE_FLOAT -DTILE=8  MatMulCudaTiled.cu -o MatMulCudaTiled_fp32_t8
!nvcc -arch=sm_75 -O3 -DUSE_FLOAT -DTILE=16 MatMulCudaTiled.cu -o MatMulCudaTiled_fp32_t16

# n=9984 builds (TILE=32 only)
!nvcc -arch=sm_75 -O3            -DTILE=32 -DN=9984 MatMulCudaTiled.cu -o MatMulCudaTiled_fp64_t32
!nvcc -arch=sm_75 -O3 -DUSE_FLOAT -DTILE=32 -DN=9984 MatMulCudaTiled.cu -o MatMulCudaTiled_fp32_t32

# Run all 6, back-to-back in the same Colab session for minimum variance
!nvprof ./MatMulCudaTiled_fp64_t8
!nvprof ./MatMulCudaTiled_fp64_t16
!nvprof ./MatMulCudaTiled_fp64_t32

!nvprof ./MatMulCudaTiled_fp32_t8
!nvprof ./MatMulCudaTiled_fp32_t16
!nvprof ./MatMulCudaTiled_fp32_t32

MatMulCudaTiled -- n=10000, dtype=double (FP64), tile=8x8
Matrix memory per buffer: 762.94 MB
Shared memory per block:  1024 bytes (2 x 8 x 8 x 8)
==17759== NVPROF is profiling process 17759, command: ./MatMulCudaTiled_fp64_t8
C[0][0]   = 60000.000000 (expected 60000.000000, abs err 0.000e+00)
C[n-1][n-1] = 60000.000000

--- Timings (ms) ---
H2D copy   :    354.335 ms
Kernel     :  10181.406 ms
D2H copy   :    541.663 ms
Total GPU  :  11077.404 ms  (11.077 s)
==17759== Profiling application: ./MatMulCudaTiled_fp64_t8
==17759== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   91.93%  10.1812s         1  10.1812s  10.1812s  10.1812s  matmul_tiled(double const *, double const *, double*, int)
                    4.88%  540.38ms         1  540.38ms  540.38ms  540.38ms  [CUDA memcpy DtoH]
                    3.19%  353.84ms         2  176.92ms  175.98ms  177.86ms  [CUDA memcpy HtoD]
      API calls:   98.33%  11.0771s    

## 5: cuBLAS —
- NVIDIA's hand-tuned BLAS library — as a practical upper bound for what a register-tiled, vendor-optimized GEMM does on T4.
- cuBLAS is column-major, our matrices are row-major. The standard trick is to compute C^T = B^T * A^T in column-major, which is bit-identical to C = A * B in row-major; for square n x n matrices with our uniform fills this is straightforward. The relevant call is at Cuda/MatMulCublas.cu lines 100-108.


In [15]:
!nvcc -arch=sm_75 -O3            MatMulCublas.cu -o MatMulCublas_fp64 -lcublas
!nvcc -arch=sm_75 -O3 -DUSE_FLOAT MatMulCublas.cu -o MatMulCublas_fp32 -lcublas

!nvprof ./MatMulCublas_fp64
!nvprof ./MatMulCublas_fp32

MatMulCublas -- n=10000, dtype=double (FP64)
Matrix memory per buffer: 762.94 MB
==21399== NVPROF is profiling process 21399, command: ./MatMulCublas_fp64
C[0][0]   = 60000.000000 (expected 60000.000000, abs err 0.000e+00)
C[n-1][n-1] = 60000.000000

--- Timings (ms, after warm-up) ---
H2D copy   :    358.063 ms
Kernel     :   7984.253 ms  (cuBLAS Dgemm)
D2H copy   :    552.508 ms
Total GPU  :   8894.824 ms  (8.895 s)
==21399== Profiling application: ./MatMulCublas_fp64
==21399== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   92.74%  16.0498s         2  8.02492s  7.98410s  8.06574s  volta_dgemm_128x64_nn
                    4.08%  706.11ms         4  176.53ms  173.54ms  183.98ms  [CUDA memcpy HtoD]
                    3.18%  550.87ms         1  550.87ms  550.87ms  550.87ms  [CUDA memcpy DtoH]
      API calls:   52.69%  9.24361s         5  1.84872s  173.79ms  8.53657s  cudaMemcpy
                   45.98%  8.06578s 

### 5.2: Nsight compute

In [16]:
!ncu --launch-skip 0 --launch-count 5 \
     --section MemoryWorkloadAnalysis \
     --section LaunchStats \
     --section Occupancy \
     ./MatMulCublas_fp32

!ncu --launch-skip 0 --launch-count 5 \
     --section MemoryWorkloadAnalysis \
     --section LaunchStats \
     --section Occupancy \
     ./MatMulCublas_fp64

MatMulCublas -- n=10000, dtype=float (FP32)
Matrix memory per buffer: 381.47 MB
==PROF== Connected to process 21551 (/content/MatMulCublas_fp32)
==PROF== Profiling "volta_sgemm_128x64_nn" - 0 (1/5): 0%....50%....100% - 8 passes
==PROF== Profiling "volta_sgemm_128x64_nn" - 1 (2/5): 0%....50%....100% - 8 passes
C[0][0]   = 60000.000000 (expected 60000.000000, abs err 0.000e+00)
C[n-1][n-1] = 60000.000000

--- Timings (ms, after warm-up) ---
H2D copy   :    225.359 ms
Kernel     :   7368.762 ms  (cuBLAS Sgemm)
D2H copy   :    313.047 ms
Total GPU  :   7907.168 ms  (7.907 s)
==PROF== Disconnected from process 21551
[21551] MatMulCublas_fp32@127.0.0.1
  volta_sgemm_128x64_nn (79, 157, 1)x(128, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Memory Workload Analysis
    ----------------- ----------- ------------
    Metric Name       Metric Unit Metric Value
    ----------------- ----------- ------------
    Memory Throughput     Gbyte/s        47.83
    Mem Busy                   